<a href="https://colab.research.google.com/github/gstephenson39/resources/blob/main/Cross_Stitch_Image_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install required libraries and dependencies.

In [1]:
pip install pillow numpy reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 21.8 MB/s eta 0:00:00


In [2]:
import os, csv, math
from PIL import Image, ImageDraw, ImageFont
import numpy as np
from reportlab.lib.pagesizes import letter, landscape
from reportlab.pdfgen import canvas
import requests, csv
from io import StringIO
from sklearn.cluster import KMeans

Full DMC table for referencing. Using CSV table from Nathan T Spencer.

In [3]:
url = "https://raw.githubusercontent.com/nathantspencer/DMC-ColorCodes/master/result.csv"
resp = requests.get(url)
data = resp.text

reader = csv.DictReader(StringIO(data))
DMC_FULL = []

for row in reader:
    try:
        # Convert code to int if possible, otherwise keep string (for B5200, Blanc, etc.)
        try:
            code = int(row["DMC_COLOR"])
        except ValueError:
            code = row["DMC_COLOR"].strip()

        # Corrected column name for COLOR_NAME (with leading space)
        name = row[" COLOR_NAME"].strip()

        # Convert hex string to RGB tuple, corrected column name for RGB_COLOR (with leading space)
        hex_rgb = row[" RGB_COLOR"].strip()
        if hex_rgb.startswith("#") and len(hex_rgb) == 7:
            r = int(hex_rgb[1:3], 16)
            g = int(hex_rgb[3:5], 16)
            b = int(hex_rgb[5:7], 16)
            rgb = (r, g, b)
        else:
            # If hex_rgb is malformed, skip this row
            continue

        DMC_FULL.append((code, name, rgb))
    except Exception as e:
        # Skip bad rows that cause other parsing issues
        continue

print(len(DMC_FULL))
print(DMC_FULL[:5])

456
[('Ecru', 'Ecru/off-white', (255, 247, 231)), (315, 'Antique Mauve - MED DK', (125, 66, 70)), ('Blanc', 'White', (238, 238, 238)), (316, 'Antique Mauve - MED', (188, 117, 127)), ('B5200', 'Snow White', (252, 252, 255))]


Create the functions to build cross stitch map and color guide from uploaded image.

In [4]:
# --- Helper functions ---

def nearest_dmc(rgb, dmc_table):
    """Return nearest DMC thread tuple (code, name, (r,g,b)) for given RGB"""
    target = np.array(rgb, dtype=float)
    dmc_rgbs = np.array([entry[2] for entry in dmc_table], dtype=float)
    distances = np.linalg.norm(dmc_rgbs - target, axis=1)
    idx = np.argmin(distances)
    return dmc_table[idx]

def hex_from_rgb(rgb):
    return '#{:02x}{:02x}{:02x}'.format(*rgb)

def choose_text_color(rgb):
    return (0,0,0) if (0.299*rgb[0]+0.587*rgb[1]+0.114*rgb[2])>160 else (255,255,255)

def generate_symbols(n):
    base = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789")
    syms = []
    while len(syms) < n:
        for c in base:
            if len(syms) >= n: break
            syms.append(c)
    return syms[:n]

def reduce_palette_kmeans(img, n_colors):
    arr = np.array(img).reshape(-1, 3)
    kmeans = KMeans(n_clusters=n_colors, random_state=42).fit(arr)
    centers = np.array(kmeans.cluster_centers_, dtype=int)

    # Map each pixel to nearest center
    new_arr = np.array([centers[np.argmin(np.linalg.norm(centers - p, axis=1))] for p in arr])
    new_img = Image.fromarray(new_arr.reshape(img.size[1], img.size[0], 3).astype(np.uint8))
    return new_img


# --- Main function ---
def make_cross_stitch_map(
    inpath,
    out_prefix,
    cols=50,
    colors=None,
    scale=20,
    pdf=True
):
    """Generate cross-stitch chart from an image without duplicate DMCs"""
    # --- Open and resize image ---
    img = Image.open(inpath).convert('RGB')
    w, h = img.size
    rows = int(round(cols * h / w))
    small = img.resize((cols, rows), resample=Image.BILINEAR)

    # Optional palette reduction
    if colors:
        small = reduce_palette_kmeans(small, colors)

    # Unique colors and counts
    arr = np.array(small)
    flat = arr.reshape(-1,3)
    uniq_cols, counts = np.unique(flat, axis=0, return_counts=True)
    idx_sorted = np.argsort(-counts)
    unique_colors_sorted = [tuple(c) for c in uniq_cols[idx_sorted]]
    counts_sorted = counts[idx_sorted]
    n_colors = len(unique_colors_sorted)
    print(f"Unique colors (sorted): {n_colors}")

    # --- Map each color -> nearest DMC + symbol, avoiding duplicates ---
    symbols = generate_symbols(n_colors)
    used_dmc_nums = set()
    color_to_dmc = {}

    for i, c in enumerate(unique_colors_sorted):
        nearest = nearest_dmc(c, DMC_FULL)
        num = nearest[0]
        if num in used_dmc_nums:
            # Find next closest DMC not already used
            target = np.array(c, dtype=float)
            available = [entry for entry in DMC_FULL if entry[0] not in used_dmc_nums]
            dmc_rgbs = np.array([entry[2] for entry in available], dtype=float)
            distances = np.linalg.norm(dmc_rgbs - target, axis=1)
            idx = np.argmin(distances)
            nearest = available[idx]
        used_dmc_nums.add(nearest[0])
        color_to_dmc[c] = (symbols[i], *nearest)

    # --- Upscale for visualization ---
    out_w, out_h = cols*scale, rows*scale
    up = small.resize((out_w, out_h), resample=Image.NEAREST)
    draw = ImageDraw.Draw(up)

    try:
        font = ImageFont.truetype("DejaVuSans-Bold.ttf", int(scale/1.8))
    except:
        font = ImageFont.load_default()

    # Draw symbols
    for y in range(rows):
        for x in range(cols):
            rgb = tuple(small.getpixel((x,y)))
            sym, num, name, dmc_rgb = color_to_dmc[rgb]
            text_color = choose_text_color(rgb)
            draw.text((x*scale + scale/3, y*scale + scale/4), sym, fill=text_color, font=font)

    # Grid lines
    for x in range(cols+1):
        draw.line([(x*scale,0),(x*scale,out_h)], fill=(0,0,0))
    for y in range(rows+1):
        draw.line([(0,y*scale),(out_w,y*scale)], fill=(0,0,0))

    out_img = f"{out_prefix}_grid.png"
    up.save(out_img)
    print("Saved image:", out_img)

    # --- CSV output ---
    csv_path = f"{out_prefix}_cells.csv"
    with open(csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['col','row','symbol','DMC_num','DMC_name','hex','r','g','b'])
        for y in range(rows):
            for x in range(cols):
                rgb = tuple(small.getpixel((x,y)))
                sym, num, name, dmc_rgb = color_to_dmc[rgb]
                writer.writerow([x+1, y+1, sym, num, name, hex_from_rgb(dmc_rgb), *dmc_rgb])
    print("Saved CSV:", csv_path)

    # --- Legend image ---
    legend_height = 30 + n_colors*25
    legend = Image.new('RGB', (300, legend_height), (255,255,255))
    ld = ImageDraw.Draw(legend)
    try:
        fontL = ImageFont.truetype("DejaVuSans.ttf", 14)
    except:
        fontL = ImageFont.load_default()

    for i, rgb in enumerate(unique_colors_sorted):
        sym, num, name, dmc_rgb = color_to_dmc[rgb]
        ld.rectangle([(10, 20+i*25),(40, 40+i*25)], fill=dmc_rgb, outline=(0,0,0))
        ld.text((50, 20+i*25), f"{sym}  DMC {num}  {name}", fill=(0,0,0), font=fontL)
    legend.save(f"{out_prefix}_legend.png")
    print("Saved legend:", f"{out_prefix}_legend.png")

    # --- PDF output ---
    if pdf:
        pdf_path = f"{out_prefix}_chart.pdf"
        c = canvas.Canvas(pdf_path, pagesize=landscape(letter))
        c.drawInlineImage(up, 20, 150, width=700, height=rows*scale*(700/(cols*scale)))
        c.setFont("Helvetica", 10)
        c.drawString(20, 120, f"Cross-Stitch Pattern: {os.path.basename(inpath)}")
        c.drawString(20, 105, f"{cols}x{rows} grid | {n_colors} colors | Scale: {scale}px")
        y = 90
        for i, rgb in enumerate(unique_colors_sorted):
            sym, num, name, dmc_rgb = color_to_dmc[rgb]
            if y < 50:
                c.showPage(); y = 550
            c.setFillColorRGB(dmc_rgb[0]/255, dmc_rgb[1]/255, dmc_rgb[2]/255)
            c.rect(20, y-5, 15, 15, fill=1)
            c.setFillColorRGB(0,0,0)
            c.drawString(45, y, f"{sym}  DMC {num}  {name}")
            y -= 18
        c.save()
        print("Saved PDF:", pdf_path)


In [11]:
from google.colab import files
uploaded = files.upload()  # choose your JPG/PNG

Saving logo_only.png to logo_only.png


Generate output from uploaded image file. Note that you should edit each line according to the commented code to ensure the output is to your specifications.

In [15]:
make_cross_stitch_map(
    inpath="logo_only.png", # use file name from import step above!
    out_prefix="BEL_logo", # change prefix for each project
    cols=35,       # width in stitches
    colors=6,     # reduce palette first (optional)
    scale=25,      # pixel size per stitch in output
    pdf=True       # create printable PDF
)

Unique colors (sorted): 6
Saved image: BEL_logo_grid.png
Saved CSV: BEL_logo_cells.csv
Saved legend: BEL_logo_legend.png
Saved PDF: BEL_logo_chart.pdf
